# Hospital Disease Analytics — Exploratory Data Analysis

**Project:** AI-Based Hospital Disease Analytics & Patient Risk Prediction
**Covers:** Module 2 (data analysis) and Module 3 (visualisation)

---

## The dataset

**Diabetes 130-US Hospitals for Years 1999-2008** — ten years of real inpatient
care from 130 US hospitals and integrated delivery networks.

| | |
|---|---|
| Source | UCI Machine Learning Repository, dataset 296 (mirrored on Kaggle) |
| Size | 101,766 encounters x 50 columns |
| Licence | CC BY 4.0 — free to use and adapt, with credit |
| DOI | 10.24432/C5230J |

> Strack B, DeShazo JP, Gennings C, Olmo JL, Ventura S, Cios KJ, Clore JN (2014).
> *Impact of HbA1c Measurement on Hospital Readmission Rates: Analysis of 70,000
> Clinical Database Patient Records.* BioMed Research International, 781670.

Cite that in your report and on your slides.

## What this notebook is for

This is the **investigation**. Before building a dashboard or training a model,
we find out what is actually in the data. Every chart ends with a statement of
*why that feature matters to the project* — a chart without an interpretation
proves nothing.

## Three facts about this data that shape everything below

1. **One row is one ENCOUNTER, not one person.** The same patient can appear
   many times. That is not duplication — they are separate admissions — but it
   means Module 4 must split train/test **by person**, or the same individual
   lands on both sides and the accuracy is inflated.
2. **There are no dates.** The data is fully de-identified. Seasonality analysis
   and time-based splitting are impossible. `encounter_seq` gives us order
   instead, which is genuine but coarser.
3. **There are no vital signs.** No blood pressure, BMI or cholesterol. `weight`
   exists but is ~97% missing. What the data *does* record richly is prior
   hospital utilisation — and that turns out to be the stronger signal anyway.

## How it fits the project

    data/raw/diabetic_data.csv
      -> ingestion/adapters/uci_diabetes.py   translate to our schema
      -> ingestion/cleaning.py                clean and engineer features
      -> MySQL                                stored via scripts/03_load_data.py
      -> analysis/metrics.py                  every statistic  <-- used below
      -> analysis/charts.py                   every figure     <-- used below
      -> THIS NOTEBOOK                        the investigation
      -> streamlit_app/                       the presentation

The notebook imports the **same** modules the dashboard imports, so a number
here can never disagree with a number there.

---
## 1. Setup

The first cell makes the project's own code importable from inside `notebooks/`.
Without it, `import hospital_analytics` fails — Python does not know to look one
folder up.

In [ ]:
import sys
from pathlib import Path

# Make src/ importable from inside notebooks/
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio

pio.renderers.default = "notebook"
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

print("pandas", pd.__version__)
print("project root:", PROJECT_ROOT)

### Cufflinks

Your brief asks for **plotly cufflinks**, which adds an `.iplot()` method
directly onto pandas DataFrames.

Be aware of what you are using: cufflinks has been unmaintained since 2021 and
calls `DataFrame.iteritems()`, which pandas **removed in version 2.0**. On any
modern install it crashes on import.

The cell below patches the missing method back before importing, then falls back
to Plotly Express if the import still fails. Recognising an abandoned dependency
and handling it is a more useful skill than pretending it works.

In [ ]:
# Cufflinks calls DataFrame.iteritems(), removed in pandas 2.0. Restore it first.
if not hasattr(pd.DataFrame, "iteritems"):
    pd.DataFrame.iteritems = pd.DataFrame.items
if not hasattr(pd.Series, "iteritems"):
    pd.Series.iteritems = pd.Series.items

try:
    import cufflinks as cf

    cf.go_offline()
    cf.set_config_file(theme="white")
    HAS_CUFFLINKS = True
    print("cufflinks ready — .iplot() available")
except Exception as exc:
    HAS_CUFFLINKS = False
    print(f"cufflinks unavailable ({type(exc).__name__}) — using Plotly Express instead")
    print("This is fine. Every chart below has a Plotly fallback.")

In [ ]:
from hospital_analytics.config import (
    DATASET_CITATION, DATASET_NAME, DATASET_PROFILE, RAW_DIR,
)
from hospital_analytics.ingestion.adapters import uci_diabetes
from hospital_analytics.ingestion.cleaning import (
    clean_medications, clean_patients, missing_value_report, outlier_report,
)
from hospital_analytics.analysis import charts, metrics

print(DATASET_NAME)
print()
for key, value in DATASET_PROFILE.items():
    print(f"  {key:<24} {value}")

---
## 2. Load the raw dataset

We read the file exactly as downloaded — no pre-cleaning. You cannot demonstrate
a cleaning pipeline on data someone else already cleaned.

If this cell fails, the dataset is not downloaded yet. Run
`python scripts/01_check_dataset.py`, which prints where to get it.

In [ ]:
raw = uci_diabetes.load_raw(RAW_DIR)
print(f"shape: {raw.shape[0]:,} rows x {raw.shape[1]} columns")
raw.head()

### What the 50 columns actually are

Worth reading properly. Most of the analytical decisions later follow from
noticing something here.

In [ ]:
print("ALL COLUMNS")
for i, col in enumerate(raw.columns, 1):
    print(f"{i:>3}. {col}")

In [ ]:
# The 23 medication columns are the thing to notice: 23 separate columns each
# holding "No"/"Steady"/"Up"/"Down". That is WIDE format, and it is unusable —
# you cannot ask "how many patients had insulin?" without naming the column.
drug_cols = [c for c in uci_diabetes.DRUG_CLASSES if c in raw.columns]
print(f"{len(drug_cols)} medication columns stored in wide format:")
print(drug_cols)
print()
print("Example values in `insulin`:")
print(raw["insulin"].value_counts())

---
## 3. Data quality assessment

**Do this before any analysis.** A statistic computed on dirty data is worse
than no statistic, because it looks authoritative.

The first thing to check in this dataset: missing values are encoded as the
string `"?"`, not as blanks. Read it naively and pandas treats them as ordinary
text, and every "missing value" count comes out as zero.

In [ ]:
# How many "?" placeholders are hiding in the file?
q = (raw == "?").sum()
q = q[q > 0].sort_values(ascending=False)
print(f"Total '?' placeholders: {int(q.sum()):,}")
print()
print(q)

In [ ]:
# Now the real missing-value picture, with "?" treated as null.
raw_nulls = raw.replace("?", np.nan)
mv = missing_value_report(raw_nulls)
mv[mv["missing"] > 0]

### The decision this forces

Three columns are badly incomplete, and they get three different treatments:

| Column | Missing | Decision | Why |
|---|---|---|---|
| `weight` | ~97% | **Drop** | Imputing 97% of a column is inventing data, not cleaning it |
| `payer_code` | ~40% | **Drop** | Not clinically meaningful for this question |
| `medical_specialty` | ~49% | **Keep as "Unknown"** | An unrecorded speciality usually means the admission had no assigned consultant — the absence is information |

Deciding what to *discard* is as much a part of the pipeline as deciding what to
keep, and being able to justify each choice separately is the point.

In [ ]:
fig = charts.missingness_bar(mv)
fig.show()

In [ ]:
# The target variable, before any transformation.
print(raw["readmitted"].value_counts())
print()
print(raw["readmitted"].value_counts(normalize=True).mul(100).round(2))

### A modelling decision, made explicit

`readmitted` has three values: `<30`, `>30` and `NO`.

Our question is specifically **30-day** readmission, so `>30` is folded into the
negative class. That is a **modelling decision, not a cleaning step** — a
readmission at 45 days is a real readmission, we are simply not predicting it.

Say this out loud in your demo. It shows you understand that the target is
something you *define*, not something you are handed.

---
## 4. Adapt and clean

Two stages, deliberately separate:

**Adapt** (`adapters/uci_diabetes.py`) — translates this dataset's vocabulary
into the project's, and splits one wide 50-column table into three normalised
tables. Supporting a different dataset later means writing one new adapter, not
editing the analysis.

**Clean** (`cleaning.py`) — dataset-agnostic quality rules.

In [ ]:
patients_raw, meds_raw, reference, adapt_report = uci_diabetes.convert(RAW_DIR)

print("ADAPTER REPORT")
for key, value in adapt_report.items():
    if key != "readmit_class_counts":
        print(f"  {key:<28} {value}")

### Why deceased patients are removed

The adapter drops encounters where the patient died or went to hospice.

They **cannot be readmitted**. Leaving them in drags the readmission rate down
for a reason that has nothing to do with care quality, and would teach a model
that "expired" predicts "not readmitted" — true, useless, and grim. Excluding
them is standard practice in the published literature.

In [ ]:
# The wide -> long medication reshape, made visible.
print("BEFORE (wide): 23 columns, one per drug")
print(raw[["encounter_id", "metformin", "insulin", "glipizide"]].head(3))
print()
print("AFTER (long): one row per prescription")
print(meds_raw.head(6).to_string(index=False))
print()
print(f"{len(meds_raw):,} prescription rows extracted")

In [ ]:
patients, report = clean_patients(patients_raw)
medications, med_report = clean_medications(meds_raw, patients["patient_id"])

print("CLEANING REPORT")
print(f"  rows in                     {report['rows_in']:,}")
print(f"  rows out                    {report['rows_out']:,}")
print(f"  columns in / out            {report['cols_in']} -> {report['cols_out']}")
print(f"  duplicate encounters        {report['duplicate_encounters_removed']:,}")
print(f"  unique people               {report['unique_people']:,}")
print(f"  repeat encounters kept      {report['repeat_encounters']:,}")
print(f"  impossible values nulled    {report['impossible_values_nulled']:,}")
print(f"  rows dropped (missing key)  {report['rows_dropped_missing_key']:,}")
print(f"  missing values remaining    {int(report['missing_after']['missing'].sum()):,}")
print(f"  30-day readmission rate     {report['readmission_rate']}%")

### Repeat patients: kept, and flagged

`repeat_encounters` is large — a substantial share of rows are return visits by
someone already in the data.

**They are not duplicates.** Each is a real, separate admission and a valid
observation. But they break the assumption that rows are independent, which
matters enormously when Module 4 splits the data: if the same person appears in
both training and test, the model can memorise them and reports an inflated
score.

So we keep them, record `person_id`, and note the constraint: **Module 4 must
group its train/test split by person.** Deleting them would throw away the
strongest signal in the dataset; ignoring the problem would produce a dishonest
accuracy figure.

In [ ]:
# Two kinds of outlier, counted separately.
outlier_report(patients_raw)

| | Meaning | Action |
|---|---|---|
| **Out of documented range** | Contradicts the dataset's own inclusion criteria — e.g. a 40-day stay when only 1-14 day stays were included. Indicates a corrupted file. | **Remove** |
| **IQR outlier** | Statistically unusual but entirely real — a patient on 70 medications with 9 diagnoses. | **Keep** |

Removing the second kind would be a serious mistake: the most complex patients
are exactly the ones the readmission model most needs to learn from.

This dataset produces very few out-of-range values, because the publishers
already applied inclusion criteria. Report that honestly rather than inflating
the number — the gate exists to catch a wrong or corrupted download.

---
## 5. Headline statistics

The numbers to memorise before your demo.

In [ ]:
k = metrics.kpis(patients)
for key, value in k.items():
    if isinstance(value, float):
        print(f"  {key:<24} {value:,.2f}")
    else:
        print(f"  {key:<24} {value:,}")

In [ ]:
metrics.describe_numeric(patients)

### Read `mean` against `50%` (the median)

Where they differ noticeably, the variable is **right-skewed** — a few very
large values pulling the average up.

That is precisely why the cleaning step imputes with the **median**. Using the
mean would systematically overstate the typical patient.

---
## 6. Univariate analysis — one variable at a time

Start by understanding each variable alone. Relationships come next.

In [ ]:
dist = metrics.disease_distribution(patients)
dist

In [ ]:
fig = charts.disease_distribution(patients)
fig.show()
print(charts.insight("disease_distribution")["why"])

In [ ]:
# Cufflinks version, if available — this is the .iplot() your brief asks for.
if HAS_CUFFLINKS:
    dist.set_index("diagnosis")["patients"].iplot(
        kind="bar", title="Encounters by diagnosis group", yTitle="Encounters",
    )
else:
    px.bar(dist, x="diagnosis", y="patients",
           title="Encounters by diagnosis group").show()

In [ ]:
fig = charts.age_histogram(patients)
fig.show()
print(charts.insight("age_histogram")["why"])

### A limitation to state up front

Age is recorded as **10-year bands** (`[70-80)`), for de-identification. We use
the band midpoint so age can be treated numerically.

That means the project's "average age" is an average of midpoints, not of true
ages. It is close enough to be useful and you should say so rather than let an
evaluator discover it.

In [ ]:
fig = charts.readmission_pie(patients)
fig.show()
print(charts.insight("readmission_pie")["why"])

In [ ]:
# THE MOST IMPORTANT NUMBER IN THE PROJECT.
rate = patients["readmitted_30d"].mean()
baseline = max(rate, 1 - rate)
print(f"30-day readmission rate      {rate * 100:.2f}%")
print(f"Majority-class baseline      {baseline * 100:.2f}% accuracy")
print()
print("A model that ALWAYS predicts 'not readmitted' scores that accuracy while")
print("catching zero at-risk patients. This is why Module 4 must be judged on")
print("RECALL and ROC-AUC, not accuracy.")

---
## 7. Bivariate analysis — what relates to readmission

Now the actual question: which patients come back?

In [ ]:
risk = metrics.risk_factor_summary(patients)
risk

In [ ]:
fig = charts.risk_factor_lift(patients)
fig.show()
print(charts.insight("risk_factors")["why"])

### How to read lift

**Lift = readmission rate WITH the factor / rate WITHOUT it.** A lift of 2.0
means the factor doubles the rate.

Lift is used rather than a correlation coefficient because a clinician and a
hospital board both understand "doubles the risk". A correlation of 0.30 means
nothing to either of them.

These measured lift values are what Module 5's GenAI explanations will be
**grounded in** — so the model cites a number from this dataset rather than
inventing a plausible-sounding statistic.

In [ ]:
fig = charts.prior_admissions_curve(patients)
fig.show()
print(charts.insight("prior_admissions")["why"])

In [ ]:
# All three kinds of prior utilisation, side by side.
for col in ["num_prior_admissions", "num_prior_emergency", "num_prior_outpatient"]:
    curve = metrics.prior_utilisation_curve(patients, col=col)
    if not curve.empty:
        print()
        print(col)
        print(curve[["label", "patients", "readmission_rate"]].to_string(index=False))

In [ ]:
by_diag = metrics.readmission_by_group(patients, "diagnosis")
by_diag

In [ ]:
fig = charts.los_box_by_diagnosis(patients)
fig.show()
print(charts.insight("los_box")["why"])

In [ ]:
fig = charts.polypharmacy_bar(patients)
fig.show()
print(charts.insight("polypharmacy")["why"])

---
### The dataset's own research question: HbA1c

This dataset was published to answer one specific question — does measuring
HbA1c during an admission relate to early readmission?

HbA1c reflects average blood sugar over roughly three months, so it says whether
a diabetic patient's condition has been **controlled**, not just what it is
today. The original study's finding was that whether the test was *ordered at
all* carries information, because ordering it signals that the team engaged with
the patient's diabetes rather than treating only what they came in for.

**Reproducing a published finding on your own pipeline is the strongest possible
evidence that your cleaning is correct.** Nothing else in your project validates
your work against peer review.

In [ ]:
a1c = metrics.a1c_testing_summary(patients)
a1c

In [ ]:
fig = charts.a1c_readmission_bar(patients)
fig.show()
print(charts.insight("a1c_testing")["why"])

In [ ]:
not_tested = a1c.loc[a1c["a1c_result"] == "Not Tested", "patients"].sum()
share = 100 * not_tested / a1c["patients"].sum()
print(f"{share:.1f}% of these diabetic admissions had NO HbA1c measurement.")
print()
print("That is the finding: the most common decision is not to test at all.")

---
### Fairness audit — deliberately before any model exists

If readmission rates differ sharply by gender within the same age band, a
carelessly trained model could encode that and distribute care unequally.

Auditing **now** and writing down the result is the point — not hoping it comes
out clean. Module 4 must repeat this check on the model's *predictions*, not
just on the input data.

In [ ]:
fig = charts.gender_age_readmission(patients)
fig.show()
print(charts.insight("gender_readmission")["why"])

In [ ]:
for col in ["gender", "race", "age_group"]:
    if col in patients.columns:
        summary = metrics.readmission_by_group(patients, col, min_count=100)
        if not summary.empty:
            print()
            print(f"--- readmission by {col} ---")
            print(summary[[col, "total_patients", "readmission_rate"]].to_string(index=False))

---
## 8. Multivariate analysis — variables together

Individual relationships are only half the picture.

In [ ]:
target_corr = metrics.target_correlations(patients)
target_corr

In [ ]:
fig = charts.target_correlation_bar(patients)
fig.show()

In [ ]:
fig = charts.correlation_heatmap(patients)
fig.show()
print(charts.insight("correlation_heatmap")["why"])

In [ ]:
# Multicollinearity check: pairs of features that measure nearly the same thing.
corr = metrics.correlation_matrix(patients)
pairs = []
for i, a in enumerate(corr.columns):
    for b in corr.columns[i + 1:]:
        r = corr.loc[a, b]
        if abs(r) >= 0.30 and "readmitted" not in (a, b):
            pairs.append({"feature_a": a, "feature_b": b, "correlation": round(r, 3)})

if pairs:
    display(pd.DataFrame(pairs).sort_values("correlation", key=abs, ascending=False))
else:
    print("No feature pairs above |r| = 0.30 — no multicollinearity problem here.")

### Why multicollinearity matters, and only sometimes

When two features carry nearly the same information, a **logistic regression**
cannot decide which deserves the credit. Its coefficients become unstable and
uninterpretable — and interpretability is the main reason to use it.

**Tree-based models (Random Forest, XGBoost) are unaffected.** They pick one at
each split and move on.

So the Module 4 decision is conditional: drop one of each redundant pair *for
the logistic regression only*, keep both for the tree models.

**A limitation to state:** correlation only detects **linear** relationships.
Features that matter only in combination will look weak here. That is part of
the argument for including XGBoost.

---
## 9. Sequence analysis — the closest thing to time

There are **no dates** in this dataset. It is fully de-identified, so monthly
trends and seasonality are not computable, and Module 4 cannot use a time-based
train/test split.

What survives is **order**: for each person, which visit this was. That is a
genuine ordering and it answers a real question — does risk climb as someone
accumulates admissions?

**Call it a sequence, never a trend.** Claiming a time axis you do not have is
the kind of thing an evaluator will catch.

In [ ]:
seq = metrics.encounter_sequence_curve(patients)
seq

In [ ]:
fig = charts.encounter_sequence_chart(patients)
fig.show()
print(charts.insight("encounter_sequence")["why"])

---
## 10. Medication analysis

This is where the wide-to-long reshape pays off. None of the questions below
could be asked of the original 23-column layout.

In [ ]:
top_meds = metrics.medication_usage(medications, top_n=15)
top_meds

In [ ]:
fig = charts.top_medications(medications)
fig.show()
print(charts.insight("medications")["why"])

In [ ]:
fig = charts.drug_class_bar(medications)
fig.show()
print(charts.insight("drug_class")["why"])

In [ ]:
# Dose CHANGES are more informative than dose levels: a change means a clinician
# actively judged the patient's control to be wrong.
changes = (
    medications.groupby("dose_status")
    .size().reset_index(name="prescriptions")
    .sort_values("prescriptions", ascending=False)
)
print(changes.to_string(index=False))
print()
adjusted = 100 * patients["medication_changed"].mean()
print(f"{adjusted:.1f}% of encounters involved a change to the diabetes regimen.")

---
## 11. Data leakage — the trap in this dataset

**Leakage** is when a feature contains information that would not be available
at the moment you actually need the prediction. The model scores brilliantly in
testing and is useless in deployment.

There is a real candidate here: `discharge_disposition`. It is strongly related
to readmission — and it is recorded **at discharge**.

If the model is meant to run *before* discharge so staff can intervene, using it
is leakage. The chart below is included for understanding the population, and
the column is flagged in `ml/__init__.py` so Module 4 makes the decision
deliberately rather than by accident.

In [ ]:
disch = metrics.discharge_destination_summary(patients)
disch

In [ ]:
fig = charts.discharge_destination_bar(patients)
fig.show()
print(charts.insight("discharge_destination")["why"])

---
## 12. Findings, and what each one decides

An EDA that ends with charts is incomplete. This is the section that turns the
investigation into design decisions.

In [ ]:
majority = max(patients["readmitted_30d"].mean(), 1 - patients["readmitted_30d"].mean())

findings = pd.DataFrame([
    {"finding": "Prior hospital utilisation is the strongest factor family",
     "evidence": "highest lift of any factor group",
     "decision": "Module 4: prior-visit counts are core features"},
    {"finding": "Rows are encounters; people repeat",
     "evidence": f"{report['repeat_encounters']:,} repeat encounters",
     "decision": "Module 4: split train/test grouped BY PERSON"},
    {"finding": "Target is heavily imbalanced",
     "evidence": f"{report['readmission_rate']}% positive class",
     "decision": "Module 4: class weights; judge on recall + ROC-AUC"},
    {"finding": "Majority-class baseline is already high",
     "evidence": f"{majority * 100:.1f}% accuracy",
     "decision": "Report the baseline alongside every model score"},
    {"finding": "No dates exist in the dataset",
     "evidence": "fully de-identified source",
     "decision": "Grouped random split, NOT a time-based split"},
    {"finding": "Discharge destination predicts strongly but leaks",
     "evidence": "recorded at discharge, not before",
     "decision": "Fix the prediction moment first, then include or exclude"},
    {"finding": "HbA1c is often not tested at all",
     "evidence": "majority of encounters untested",
     "decision": "Treat 'Not Tested' as its own category, never as missing"},
    {"finding": "weight is ~97% missing",
     "evidence": "missing-value report",
     "decision": "Drop the column; do not impute"},
])
findings

### The five things to be able to say without notes

1. **Prior hospital use dominates.** How much care someone already used predicts
   future use better than any single clinical measurement in this dataset.
2. **Rows are encounters, not people.** This single fact determines how Module 4
   must be evaluated, and getting it wrong inflates every score.
3. **The baseline is the headline.** Always guessing "not readmitted" scores high
   accuracy while helping nobody. Recall and ROC-AUC are the honest metrics.
4. **Absence is data.** An HbA1c that was never ordered, a speciality never
   recorded — those tell you about the care process, not about a broken file.
5. **Leakage is a design decision, not a bug to find later.** Fix *when* the
   prediction happens, then choose features available at that moment.

### What Module 5 needs from this notebook

The GenAI report generator will use **grounded generation**: rather than asking
a model to explain a prediction from its own medical knowledge — where it will
produce a fluent, confident, invented statistic — we supply the measured lift
values computed above and instruct it to use only those.

`metrics.risk_factor_summary()` already computes exactly the facts the prompt
needs. That is why Module 2 had to come before Module 5.

In [ ]:
# Save the cleaned tables so the dashboard can start instantly.
from hospital_analytics.config import PROCESSED_MEDS_CSV, PROCESSED_PATIENTS_CSV

patients.to_csv(PROCESSED_PATIENTS_CSV, index=False)
medications.to_csv(PROCESSED_MEDS_CSV, index=False)
print(f"Saved {len(patients):,} rows to {PROCESSED_PATIENTS_CSV}")
print(f"Saved {len(medications):,} rows to {PROCESSED_MEDS_CSV}")
print()
print("Next:  streamlit run streamlit_app/app.py")
print()
print("Cite in your report:")
print(f"  {DATASET_CITATION}")